# Análisis COVID-19 en Perú y Latinoamérica

Exploración de datos públicos de Our World in Data (OWID) para entender el impacto del COVID-19 en Perú comparado con países de la región.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from io import StringIO
import requests
import warnings
warnings.filterwarnings("ignore")

In [ ]:
url = "https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv"
resp = requests.get(url, timeout=60)
df_full = pd.read_csv(StringIO(resp.text))

paises_latam = ["Peru", "Brazil", "Chile", "Colombia", "Argentina", "Mexico", "Ecuador", "Bolivia"]
latam = df_full[df_full["location"].isin(paises_latam)].copy()
peru = latam[latam["location"] == "Peru"].copy()
peru["date"] = pd.to_datetime(peru["date"])
peru["anio"] = peru["date"].dt.year
peru["mes"] = peru["date"].dt.month
latam["date"] = pd.to_datetime(latam["date"])

print(f"Registros Perú: {len(peru):,}")
print(f"Registros LATAM: {len(latam):,}")
print(f"Rango: {peru['date'].min().date()} → {peru['date'].max().date()}")

## Resumen estadístico

In [ ]:
cols_interes = ["new_cases", "new_deaths", "new_cases_smoothed", "total_vaccinations_per_hundred"]
peru[cols_interes].describe().round(1)

## 1. Casos diarios en Perú con olas pandémicas

In [ ]:
paleta = ["#1B4965", "#5FA8D3", "#62B6CB", "#BEE9E8", "#CAE9FF"]
colores_olas = ["#FFE0B2", "#FFCCBC", "#D1C4E9", "#B2DFDB", "#C8E6C9"]

serie = peru.dropna(subset=["new_cases_smoothed"]).copy()
umbral = 5000

serie["en_ola"] = serie["new_cases_smoothed"] > umbral
serie["cambio_ola"] = serie["en_ola"].ne(serie["en_ola"].shift()).cumsum()
olas = serie[serie["en_ola"]].groupby("cambio_ola")["date"].agg(["min", "max"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(serie["date"], serie["new_cases_smoothed"], alpha=0.7, color=paleta[0], linewidth=0)
ax.plot(serie["date"], serie["new_cases_smoothed"], color=paleta[0], linewidth=0.6)

for i, (_, row) in enumerate(olas.iterrows()):
    color_idx = i % len(colores_olas)
    ax.axvspan(row["min"], row["max"], alpha=0.3, color=colores_olas[color_idx], label=f"Ola {i+1}")

ax.axhline(y=umbral, color="#BF360C", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(serie["date"].iloc[10], umbral + 500, f"Umbral: {umbral:,}", fontsize=8, color="#BF360C")

ax.set_title("Casos diarios suavizados en Perú", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Casos (promedio 7 días)")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2. Muertes por millón en países LATAM

In [ ]:
ultimo_registro = latam.sort_values("date").groupby("location").last().reset_index()
muertes_millon = (
    ultimo_registro[["location", "total_deaths_per_million"]]
    .dropna()
    .sort_values("total_deaths_per_million", ascending=True)
)

colores_barra = ["#1B4965" if loc == "Peru" else "#BEE9E8" for loc in muertes_millon["location"]]

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=muertes_millon, x="total_deaths_per_million", y="location",
    palette=colores_barra, edgecolor="none", ax=ax
)

for i, (_, row) in enumerate(muertes_millon.iterrows()):
    ax.text(row["total_deaths_per_million"] + 50, i, f"{row['total_deaths_per_million']:,.0f}",
            va="center", fontsize=9, color="#333")

ax.set_title("Muertes totales por millón de habitantes", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Muertes por millón")
ax.set_ylabel("")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 3. Casos nuevos vs progreso de vacunación

In [ ]:
peru_vac = peru.dropna(subset=["new_cases_smoothed"]).copy()

fig, ax1 = plt.subplots(figsize=(14, 5))

color_casos = "#1B4965"
ax1.plot(peru_vac["date"], peru_vac["new_cases_smoothed"], color=color_casos, linewidth=1.2, label="Casos diarios (suavizados)")
ax1.fill_between(peru_vac["date"], peru_vac["new_cases_smoothed"], alpha=0.15, color=color_casos)
ax1.set_ylabel("Casos diarios", color=color_casos, fontsize=11)
ax1.tick_params(axis="y", labelcolor=color_casos)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))

ax2 = ax1.twinx()
color_vac = "#E65100"
ax2.plot(peru_vac["date"], peru_vac["people_vaccinated_per_hundred"], color=color_vac, linewidth=2, linestyle="-", label="Vacunados (%)")
ax2.set_ylabel("Población vacunada (%)", color=color_vac, fontsize=11)
ax2.tick_params(axis="y", labelcolor=color_vac)
ax2.set_ylim(0, 100)

lineas1, etiquetas1 = ax1.get_legend_handles_labels()
lineas2, etiquetas2 = ax2.get_legend_handles_labels()
ax1.legend(lineas1 + lineas2, etiquetas1 + etiquetas2, loc="upper left", fontsize=9, framealpha=0.9)

ax1.set_title("Perú: Nuevos casos vs avance de vacunación", fontsize=14, fontweight="bold", pad=12)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Evolución interactiva: Perú vs principales países LATAM

In [ ]:
top_paises = (
    latam.sort_values("date").groupby("location")["total_cases"].last()
    .nlargest(3).index.tolist()
)
if "Peru" not in top_paises:
    top_paises.append("Peru")

df_top = latam[latam["location"].isin(top_paises)].copy()
df_top = df_top.dropna(subset=["new_cases_smoothed"])

colores_plotly = {"Peru": "#1B4965", "Brazil": "#5FA8D3", "Argentina": "#62B6CB",
                  "Colombia": "#BEE9E8", "Mexico": "#CAE9FF", "Chile": "#90A4AE"}

fig = px.line(
    df_top, x="date", y="new_cases_smoothed", color="location",
    color_discrete_map=colores_plotly,
    labels={"new_cases_smoothed": "Casos (prom. 7d)", "date": "Fecha", "location": "País"},
    title="Casos diarios suavizados: Perú vs principales países LATAM"
)
fig.update_layout(
    xaxis=dict(rangeslider=dict(visible=True), type="date"),
    template="plotly_white",
    font=dict(family="Inter, sans-serif"),
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.show()

## 5. Tasa de letalidad mensual (CFR)

In [ ]:
peru_mensual = peru.copy()
peru_mensual["periodo"] = peru_mensual["date"].dt.to_period("M")

cfr_mensual = (
    peru_mensual.groupby("periodo")
    .agg(casos=("new_cases", "sum"), muertes=("new_deaths", "sum"))
    .reset_index()
)
cfr_mensual["cfr_pct"] = (cfr_mensual["muertes"] / cfr_mensual["casos"].replace(0, np.nan)) * 100
cfr_mensual["fecha_str"] = cfr_mensual["periodo"].astype(str)
cfr_mensual = cfr_mensual.dropna(subset=["cfr_pct"])

fig = go.Figure()
fig.add_trace(go.Bar(
    x=cfr_mensual["fecha_str"], y=cfr_mensual["cfr_pct"],
    marker_color="#1B4965",
    hovertemplate="%{x}<br>CFR: %{y:.2f}%<br>Casos: %{customdata[0]:,}<br>Muertes: %{customdata[1]:,}<extra></extra>",
    customdata=cfr_mensual[["casos", "muertes"]].values
))

fig.update_layout(
    title="Tasa de letalidad mensual (CFR) en Perú",
    xaxis_title="Mes",
    yaxis_title="CFR (%)",
    template="plotly_white",
    font=dict(family="Inter, sans-serif"),
    height=450,
    xaxis=dict(tickangle=-45, dtick=3)
)
fig.show()

## Hallazgos principales

- Perú registra la mayor tasa de muertes por millón en Latinoamérica
- Se identifican múltiples olas pandémicas con picos diferenciados
- El avance de vacunación coincide con la reducción progresiva de casos severos
- La tasa de letalidad fue más alta al inicio de la pandemia y disminuyó con la vacunación y mejoras en tratamiento